Copyright 2026 Google LLC

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a target="_blank" href="https://colab.research.google.com/github/google-gemma/gbench/blob/main/examples/notebooks/05_agentic_quality_with_gemmaclaw.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# gbench agentic quality simulation with GemmaClaw

**Author:** [Luciano Martins](https://github.com/lucianommartins)

This notebook demonstrates how to evaluate multi-turn autonomous agent workflows using `gbench --quality-only` and the GemmaClaw QA microservice (`48+ scenarios`). You will learn how `gbench` provisions the TypeScript simulation harness (cloned from the `gemmaclaw` remote), manages `.buildstamp` compilation caching (~1-2 minutes initial compile), and tests session recall and tool routing against an OpenAI-compatible `/v1` endpoint.

> **Note:** the `gemma-4` GGUF (`unsloth/gemma-4-E4B-it-qat-GGUF`) and tokenizer (`google/gemma-4-E4B-it`) names used below are the intended Gemma 4 launch artifacts and are placeholders until the model is public.

## Serving paths (this notebook shows both)

* **Ollama local serving (featured here):** serve the `gemma-4` QAT GGUF with Ollama and point `gbench` at its OpenAI endpoint `http://localhost:11434/v1`. All cells below use this path.
* **vLLM remote endpoint (alternative):** if you already serve `gemma-4` with vLLM, skip the Ollama setup and run `gbench` against it directly, e.g. `--remote-endpoint http://127.0.0.1:8000/v1 --tokenizer google/gemma-4-26B-A4B-it` (served model id `google/gemma-4-26B-A4B-it`).

## Learning objectives

1. Configure Ollama to serve a quantized Google Gemma 4 model (`unsloth/gemma-4-E4B-it-qat-GGUF`) with a context window of 8192 tokens.
2. Understand GemmaClaw microservice provisioning and TypeScript `.buildstamp` caching.
3. Execute multi-turn agentic simulation scenarios (`--quality-only`) over OpenAI-compatible `/v1` endpoints.
4. Filter specific capability scenarios using `--scenarios memory/session_recall.json plugins/mcp_routing.json`.
5. Perform a clean session shutdown to terminate background servers and reclaim hardware memory.

## Useful resources

* [gbench GitHub repository](https://www.github.com/google-gemma/gbench)
* [Ollama documentation](https://github.com/ollama/ollama)
* [GemmaClaw agent simulation harness](https://github.com/gemmaclaw/gemmaclaw)

## 1. Environment setup and installation

We clone the `gbench` repository from GitHub, change directory into the project root (`%cd gbench`), and install the package in editable mode (`%pip install -e .`). This builds and links the `gbench` CLI executable without installing unnecessary development linters.

In [ ]:
import os, sys
from pathlib import Path

# Safe environment setup: Always normalize to top-level repository
if Path("/content").exists():
    %cd -q /content
    if not Path("/content/gbench").is_dir():
        !git clone https://github.com/google-gemma/gbench.git
    %cd -q /content/gbench
else:
    if not Path("pyproject.toml").is_file() and not Path("gbench").is_dir():
        if not Path("gbench").is_dir():
            !git clone https://github.com/google-gemma/gbench.git
        %cd gbench

%pip install -e . -q
import gbench
print(f"gbench version {gbench.__version__} installed successfully.")

# Inspect available evaluation pillars
!gbench --list pillars

## Hugging Face authentication (required)

This notebook downloads the Gemma 4 GGUF (and its vision projector) plus the `google/gemma-4-*` tokenizer from the Hugging Face Hub with `huggingface_hub` — some are **gated** — so an **`HF_TOKEN` is required**.

1. Create a **read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and accept the license on any gated model page you use.
2. On **Colab**: click the **🔑 key icon (Secrets)** in the left sidebar → **Add new secret**, name it `HF_TOKEN`, paste the token, and toggle **Notebook access** on.
3. **Elsewhere**: set it in your environment, e.g. `export HF_TOKEN=hf_...` (or `os.environ["HF_TOKEN"] = "hf_..."`).

The next cell loads the token and stops with instructions if it is missing.

In [ ]:
import os

# HF_TOKEN is REQUIRED: this notebook downloads the Gemma 4 GGUF (+ vision projector)
# and the google/gemma-4-* tokenizer from the Hugging Face Hub via huggingface_hub,
# which authenticates with it (resumable, higher rate limits, and access to gated repos).
try:
    from google.colab import userdata          # Colab: read from the Secrets vault
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass                                        # not on Colab, or the secret is unset
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError(
        "HF_TOKEN is not set. On Colab: click the key icon (Secrets) in the left "
        "sidebar, add a secret named HF_TOKEN, and turn on notebook access. "
        "Elsewhere: os.environ['HF_TOKEN'] = 'hf_...'. "
        "Create a read token at https://huggingface.co/settings/tokens."
    )
print("HF_TOKEN loaded.")

## 2. Installing Ollama locally

We check if the Ollama binary is present on the system. If it is not found, we install Ollama using its official Linux installation script (`curl -fsSL https://ollama.com/install.sh | sh`). Finally, we run `ollama --version` to verify that the installation succeeded and the CLI is available.

In [ ]:
import subprocess, os, shutil, glob

# (Re)install Ollama unless BOTH the binary and its llama-server runner are present.
# A binary-only partial install fails every request with "llama-server binary not
# found", so checking only for the binary would skip the repair.
def _ollama_ready():
    if not shutil.which("ollama"):
        return False
    return any(glob.glob(p) for p in (
        "/usr/local/lib/ollama/llama-server",
        "/usr/local/lib/ollama/*/llama-server",
        "/usr/lib/ollama/llama-server",
    ))

if not _ollama_ready():
    print("Installing/repairing Ollama (binary + llama-server runner)...")
    # Ensure zstd is available (required by Ollama Linux tar.zst packages)
    subprocess.run("command -v zstd >/dev/null || (command -v apt-get >/dev/null && apt-get update -qq && apt-get install -y -qq zstd)", shell=True)
    # Run official Ollama installer
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
else:
    print("Ollama (with llama-server runner) already installed.")

# Ensure binary directory is present in PATH for subsequent cells
for p in ["/usr/local/bin", "/usr/bin", os.path.expanduser("~/.local/bin")]:
    if os.path.exists(os.path.join(p, "ollama")) and p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"

!ollama --version

## 3. Launching background Ollama server

We launch the `ollama serve` process in the background and send a health check request to `http://localhost:11434/` to verify that the HTTP API is alive ("Ollama is running").

In [ ]:
import subprocess, time, requests
try:
    resp = requests.get("http://localhost:11434/", timeout=2)
    print("Ollama server already active:", resp.text.strip())
except Exception:
    print("Starting background ollama serve...")
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)
    resp = requests.get("http://localhost:11434/")
    print("Server health check:", resp.text.strip())

## 4. Downloading the GGUF and writing the Modelfile

We download the quantized Gemma 4 GGUF **and its vision projector (`mmproj`)** from the Hugging Face Hub with `huggingface_hub` (authenticated via `HF_TOKEN`, so the download is resumable and not rate-limited), then write an Ollama `Modelfile.qat` that points `FROM` the **local** files:
* **`num_ctx 8192`**: Context window of 8192 tokens.
* **`SYSTEM prompt`**: System instruction defining Gemma 4 AI assistant capabilities.

We download here rather than letting `ollama create` pull `hf.co/…` itself: Ollama's puller is anonymous (it can't use `HF_TOKEN`) and can stall on the HF CDN. The two-`FROM` import (main GGUF + `mmproj`) keeps the model's **vision** capability.

In [ ]:
import os
from huggingface_hub import HfApi, hf_hub_download

HF_REPO = "unsloth/gemma-4-E4B-it-qat-GGUF"
HF_QUANT = "UD-Q4_K_XL"
token = os.environ["HF_TOKEN"]  # required; loaded in the Hugging Face auth cell above

# Download the model GGUF and its vision projector (mmproj) via huggingface_hub, which
# authenticates with HF_TOKEN - Ollama's own hf.co puller is anonymous and can stall
# on the HF CDN. Build the model FROM the local files: a two-FROM import (main +
# mmproj) keeps gemma-4's vision capability (verified with `ollama show`). realpath
# resolves the HF cache symlink so `ollama create` reads the actual file.
files = HfApi().list_repo_files(HF_REPO, token=token)
main = [f for f in files if f.endswith(".gguf") and HF_QUANT in f]
proj = [f for f in files if f.endswith(".gguf") and "mmproj" in f.lower() and "-F16" in f]
if not main:
    raise RuntimeError(f"No {HF_QUANT} .gguf found in {HF_REPO}.")
GGUF_PATH = os.path.realpath(hf_hub_download(HF_REPO, main[0], token=token))
lines = [f"FROM {GGUF_PATH}"]
if proj:  # vision projector -> keeps multimodal capability
    lines.append(f"FROM {os.path.realpath(hf_hub_download(HF_REPO, proj[0], token=token))}")
lines += ['PARAMETER num_ctx 8192',
          'SYSTEM "You are a helpful Gemma 4 AI assistant with reasoning, vision, and tool calling capabilities."']
with open("Modelfile.qat", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print("Created Modelfile.qat from local GGUF" + (" + mmproj (vision)" if proj else ""))

## 5. Registering model and running generation smoke test

We register our custom model tag (`gemma4-qat:4b`) with `ollama create -f Modelfile.qat`. Because `Modelfile.qat` points `FROM` the local GGUF (and `mmproj`) downloaded in the previous cell, this reads from disk — no network pull. We then run a quick generation test to verify the model loads into hardware memory and generates tokens correctly.

In [ ]:
import subprocess, requests

MODEL_TAG = "gemma4-qat:4b"
print(f"Registering model {MODEL_TAG} from the local GGUF (built in the previous cell)...")
subprocess.run(["ollama", "create", MODEL_TAG, "-f", "Modelfile.qat"], check=True)

print("Running quick generation smoke test via Ollama API (cold load into GPU VRAM)...")
resp = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": MODEL_TAG, "prompt": "Reply with the single word: READY.", "stream": False},
    timeout=300,
)
print("Smoke test response:", resp.json().get("response", "").strip())

## 6. Verifying OpenAI REST endpoint readiness

Before launching `gbench`, we query `http://localhost:11434/v1/models` to verify that Ollama is serving standard OpenAI `/v1` REST payloads and that our registered model is listed.

In [ ]:
import requests
resp = requests.get("http://localhost:11434/v1/models")
print("OpenAI /v1/models endpoint HTTP status:", resp.status_code)
models = [m["id"] for m in (resp.json().get("data") or [])]
print("Available REST models:", models)
if not models:
    print("No models registered yet - re-run the model registration cell above.")

## 7. Running autonomous agent quality simulation

We can inspect all capability pillars using `!gbench --list pillars`.

We execute `gbench --quality-only` to evaluate multi-turn autonomous agent workflows. During its initial invocation, `gbench` provisions the GemmaClaw TypeScript simulation harness (cloned from the `gemmaclaw` remote) and compiles the `.buildstamp` cache (~1-2 minutes). Once cached, subsequent runs execute immediately against the Ollama `/v1` endpoint.

The cell below runs against the Ollama endpoint. To use a **vLLM** server instead, point `--remote-endpoint` at it and pass the matching tokenizer, e.g.:

```bash
python -m gbench --quality-only \
        --remote-endpoint http://127.0.0.1:8000/v1 \
        --tokenizer google/gemma-4-26B-A4B-it \
        --results-dir ./results_quality
```

With `--remote-endpoint`, `gbench` auto-detects the served model id from `/v1/models` (so `--models` is optional here).

In [ ]:
# List all benchmark pillars
!gbench --list pillars

# Execute autonomous agent quality evaluation
!gbench --quality-only \
        --models gemma4-qat:4b \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --results-dir ./results_quality

## 8. Analyzing multi-turn session recall and tool routing

`gbench --quality-only` writes its result to `<results-dir>/quality/quality_<model>_<format>.json` (for the run above: `./results_quality/quality/quality_<served-model>_remote-endpoint.json`). We load that file to inspect how accurately the QAT model maintains long-context session recall across multiple agent turns and routes structured tool invocations.

In [ ]:
import glob, json

# Quality results land in <results-dir>/quality/quality_<model>_<format>.json.
# The served-model tag can contain characters like ':', so glob for the file.
paths = sorted(glob.glob("./results_quality/quality/quality_*.json"))
if not paths:
    print("No quality result file found under ./results_quality/quality/ - run the cell above first.")
else:
    result_path = paths[-1]
    print(f"Loading {result_path}")
    data = json.load(open(result_path))
    print(f"Model:      {data.get('model_name')} ({data.get('format')})")
    print(f"Pass rate:  {data.get('pass_rate'):.1f}%  "
          f"({data.get('passed_scenarios')}/{data.get('total_scenarios')} scenarios passed)")
    print(f"Failed:     {data.get('failed_scenarios')}   Blocked: {data.get('blocked_scenarios')}")

In [ ]:
!gbench --quality-only \
        --models gemma4-qat:4b \
        --scenarios memory/session_recall.json plugins/mcp_routing.json \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --results-dir ./results_quality_filtered

## 9. Filtering specific capability scenarios

You can filter the 48+ simulation scenarios using `--scenarios memory/session_recall.json plugins/mcp_routing.json` to focus on specific agent capabilities.

## 10. Session cleanup and server shutdown

We terminate background Ollama server processes and remove temporary Modelfiles.

In [ ]:
import subprocess, os

subprocess.run(["pkill", "-f", "ollama"], check=False)
if os.path.exists("Modelfile.qat"):
    os.remove("Modelfile.qat")
print("Session cleanup complete.")